# NB06 · Semana 14 — Serving: gold y features en PostgreSQL + tablero en Metabase

**Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07** · Base de la **Entrega 4 (E4)**

## Objetivo

Cerrar el pipeline: la gold y los features llegan a PostgreSQL y desde ahí los lee Metabase.

```
bronze (Delta, MinIO) ─▶ dbt: stg_urlhaus ─▶ fct_familias_mensual ─▶ fct_amenazas_mensual
                                                      └─▶ fct_features_familias
                                   │
            DAG pipeline_grupo07 · tarea publicar_gold (todos los días, una transacción)
                                   ▼
            PostgreSQL · warehouse.grupo07.*  ─▶  Metabase
```

**La publicación la hace el DAG, no este notebook.** Acá se verifica lo que quedó publicado, se retira la vista vieja y se deja la guía del tablero.

**¿Por qué mover la gold del lakehouse al warehouse?** DuckDB corre dentro de un proceso y lee archivos del lake: no atiende a varios usuarios a la vez ni maneja permisos. PostgreSQL sí: Metabase se conecta con el usuario del grupo, ve solo el esquema `grupo07` y lee tablas chicas, ya agregadas, que responden rápido.

In [ ]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import PG, info, pg_url
info()

In [ ]:
# Dependencias: el JupyterHub puede reiniciarse y perder lo instalado con pip
%pip install -q -r {os.path.join(_d, 'requirements.txt')}

## 1. Qué publicó el DAG

Las tres tablas tienen que existir, no estar vacías y **sumar la misma cantidad de URLs**: todas salen de la misma silver (lo verifica también el test `assert_gold_cuadra` antes de publicar).

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

ESQUEMA = os.environ.get("PG_SCHEMA") or PG["user"]          # grupo07
TABLAS = ["fct_familias_mensual", "fct_amenazas_mensual", "fct_features_familias"]
engine = create_engine(pg_url())

filas = []
with engine.connect() as c:
    for t in TABLAS:
        n, urls, desde, hasta, publicado = c.execute(text(
            f'select count(*), sum(cantidad_amenazas), min(make_date(anio, mes, 1)), '
            f'max(make_date(anio, mes, 1)), max(actualizado_en) from "{ESQUEMA}".{t}')).one()
        filas.append({"tabla": t, "filas": n, "urls": int(urls), "desde": desde, "hasta": hasta,
                      "publicado": publicado})
publicado = pd.DataFrame(filas)
print(publicado.to_string(index=False))
assert (publicado.filas > 0).all(), "Hay una tabla vacía"
assert publicado.urls.nunique() == 1, "Las tablas no suman lo mismo"
print(f"\n✅ Las tres tablas suman {publicado.urls.iloc[0]} URLs")

## 2. Por qué la publicación no duplica

`publicar_gold` hace `TRUNCATE` + `INSERT` de cada tabla **dentro de una sola transacción**: si algo falla a mitad de camino, PostgreSQL vuelve a la versión anterior completa. Correrla dos veces deja exactamente lo mismo (lo probamos publicando dos veces seguidas contra un PostgreSQL de prueba). Además cada tabla tiene **clave primaria** en su grano, así que una fila duplicada ni siquiera se puede insertar:

In [ ]:
with engine.connect() as c:
    claves = c.execute(text("""
        select tc.table_name, string_agg(kcu.column_name, ', ' order by kcu.ordinal_position)
        from information_schema.table_constraints tc
        join information_schema.key_column_usage kcu
          on kcu.constraint_name = tc.constraint_name and kcu.table_schema = tc.table_schema
        where tc.table_schema = :e and tc.constraint_type = 'PRIMARY KEY'
        group by 1 order by 1"""), {"e": ESQUEMA}).fetchall()
for tabla, cols in claves:
    print(f"{tabla:24} clave primaria: ({cols})")

## 3. Features

Están definidos **una sola vez**, en el modelo dbt `fct_features_familias` (lo escribe NB05, sección 3c). Grano: una fila por mes y familia de malware, con los meses sin actividad completados con 0.

| Feature | Qué es |
|---|---|
| `cantidad_mes_anterior` | URLs de la familia el mes calendario anterior |
| `variacion_mensual_pct` | cambio % contra el mes anterior (vacío si el anterior fue 0) |
| `promedio_movil_3m` | promedio del mes y los dos anteriores |
| `acumulado` | total desde la primera aparición de la familia |
| `pct_del_mes` | participación de la familia en las URLs del mes |
| `mes_completo` | falso para el último mes, que puede estar incompleto |

**Sin fuga temporal:** todas las ventanas miran solo hacia atrás (`lag`, `rows between 2 preceding and current row`, `unbounded preceding`). El feature de marzo usa marzo, febrero y enero, nunca abril. El test `assert_features_sin_fuga` los recalcula desde cero filtrando `fecha <= mes del feature` y falla si no coinciden; lo probamos cambiando la ventana a `1 following` y el test detectó la fuga.

**Reproducibles:** es SQL versionado que se recalcula entero desde la gold en cada corrida: con los mismos datos, da lo mismo.

Las familias más activas del último mes completo y cómo vienen contra el mes anterior. Muchas familias de URLhaus llegan por **campañas**: aparecen fuerte un mes y desaparecen (LuaJIT-loader tuvo 4.623 URLs en junio, 0 en julio–septiembre y 13.762 en octubre). Por eso se ordena por volumen y se marca aparte la familia que el mes anterior tuvo menos de 10 URLs: con una base tan chica la variación % no dice nada (SalatStealer pasó de 2 a 7.820, +390.900 %).

In [ ]:
with engine.connect() as c:
    crecen = pd.read_sql(text(f"""
        select familia_malware, fecha, cantidad_amenazas, cantidad_mes_anterior,
               variacion_mensual_pct, promedio_movil_3m, pct_del_mes,
               case when cantidad_mes_anterior < 10 then 'nueva o reaparece'
                    when variacion_mensual_pct >= 50 then 'crece fuerte'
                    when variacion_mensual_pct > 0 then 'crece'
                    else 'baja' end as situacion
        from "{ESQUEMA}".fct_features_familias
        where fecha = (select max(fecha) from "{ESQUEMA}".fct_features_familias where mes_completo)
          and familia_malware <> 'sin_familia' and cantidad_amenazas > 0
        order by cantidad_amenazas desc
        limit 15"""), c)
crecen

## 4. Retirar la vista vieja

Antes los features eran una vista (`v_amenazas_features`) que creaba este notebook a mano, sin tests y calculada por tipo de amenaza (URLhaus publica un solo tipo, así que `pct_del_mes` daba siempre 100 %). La reemplaza `fct_features_familias`, que viene de dbt. Se borra para que no queden dos definiciones de los mismos features.

In [ ]:
with engine.begin() as c:
    c.execute(text(f'drop view if exists "{ESQUEMA}".v_amenazas_features'))
print("Vista vieja retirada (si existía).")

## 5. Documentación dentro del warehouse

`publicar_gold` copia las descripciones de `schema.yml` como `COMMENT` de PostgreSQL: Metabase las muestra al lado de cada columna, y la documentación se escribe una sola vez (en dbt).

In [ ]:
with engine.connect() as c:
    comentarios = pd.read_sql(text("""
        select c.relname as tabla, a.attname as columna, col_description(a.attrelid, a.attnum) as descripcion
        from pg_attribute a
        join pg_class c on c.oid = a.attrelid
        join pg_namespace n on n.oid = c.relnamespace
        where n.nspname = :e and c.relname = 'fct_features_familias' and a.attnum > 0 and not a.attisdropped
        order by a.attnum"""), c, params={"e": ESQUEMA})
pd.set_option("display.max_colwidth", 120)
comentarios

## 6. Tablero en Metabase

En **https://metabase.nexoacademia.org**, base `warehouse`, esquema `grupo07`. Como no somos admin, si las tablas nuevas no aparecen en *Explorar datos*, usar **Nuevo → Consulta SQL** con las consultas de abajo.

Dashboard: **"Amenazas URLhaus — grupo07"**. Cada gráfico responde una pregunta, y el título del gráfico es la pregunta. Todas son *Nuevo → Consulta SQL* con la base **warehouse** elegida arriba a la izquierda (no "Sample Database").

**1 · ¿Cuántas URLs maliciosas nuevas se registraron el último mes completo, y cómo viene contra el anterior?** — visualización *Tendencia* (número grande con la variación)
```sql
select fecha as mes, sum(cantidad_amenazas) as urls_nuevas
from grupo07.fct_features_familias
where mes_completo
group by 1 order by 1
```

**2 · ¿Cómo evolucionó la cantidad de URLs nuevas por mes en el último año y medio?** — *Línea* (el último mes está en curso)
```sql
select fecha as mes, sum(cantidad_amenazas) as urls_nuevas
from grupo07.fct_features_familias
where fecha >= (select max(fecha) from grupo07.fct_features_familias) - interval '17 months'
group by 1 order by 1
```

**3 · ¿Qué familias de malware explican el volumen de cada mes?** — *Barras apiladas* (eje X: mes; series: familia). Muestra las campañas: una familia que aparece fuerte un mes y desaparece.
```sql
with ult as (select max(fecha) as f from grupo07.fct_features_familias where mes_completo),
ventana as (
    select ff.* from grupo07.fct_features_familias ff, ult
    where ff.fecha > ult.f - interval '6 months' and ff.fecha <= ult.f
),
top as (
    select familia_malware from ventana where familia_malware <> 'sin_familia'
    group by 1 order by sum(cantidad_amenazas) desc limit 6
)
select v.fecha as mes,
       case when v.familia_malware in (select familia_malware from top) then v.familia_malware
            when v.familia_malware = 'sin_familia' then 'sin familia identificada'
            else 'otras familias' end as familia,
       sum(v.cantidad_amenazas) as urls
from ventana v
group by 1, 2
having sum(v.cantidad_amenazas) > 0
order by 1, 3 desc
```

**4 · Entre las familias ya establecidas, ¿cuáles crecieron y cuáles cayeron más el último mes?** — *Barras horizontales* (variación % contra el mes anterior; solo familias con al menos 30 URLs el mes anterior, para no confundir ruido con tendencia)
```sql
with ult as (select max(fecha) as f from grupo07.fct_features_familias where mes_completo),
base as (
    select ff.familia_malware as familia, ff.variacion_mensual_pct as variacion_pct
    from grupo07.fct_features_familias ff, ult
    where ff.fecha = ult.f and ff.cantidad_mes_anterior >= 30 and ff.familia_malware <> 'sin_familia'
      and ff.variacion_mensual_pct is not null
),
r as (
    select *, row_number() over (order by variacion_pct desc) as sube,
              row_number() over (order by variacion_pct asc) as baja
    from base
)
select familia, variacion_pct from r
where sube <= 5 or baja <= 5
order by variacion_pct desc
```

**5 · ¿Con qué tipo de archivo se distribuye cada familia?** — *Barras apiladas al 100 %* (una barra por familia; las 8 más activas de los últimos 6 meses)
```sql
with desde as (select max(fecha) - interval '6 months' as f from grupo07.fct_familias_mensual),
top as (
    select fm.familia_malware from grupo07.fct_familias_mensual fm, desde
    where fm.familia_malware <> 'sin_familia' and fm.fecha > desde.f
    group by 1 order by sum(fm.cantidad_amenazas) desc limit 8
)
select f.familia_malware as familia,
       case when f.tipo_archivo = 'desconocido' then 'sin dato' else f.tipo_archivo end as tipo_de_archivo,
       sum(f.cantidad_amenazas) as urls
from grupo07.fct_familias_mensual f
join top using (familia_malware), desde
where f.fecha > desde.f
group by 1, 2 order by 1, 3 desc
```

**Armado sugerido:** arriba, el 1 (chico) y una tarjeta de texto con qué es el dataset, la fuente (URLhaus, abuse.ch) y que se actualiza a diario; al medio, el 2 a lo ancho; abajo, el 3 y el 4 lado a lado, y el 5 a lo ancho.

**Qué decisión se toma con esto:** un equipo de seguridad ve si el volumen sube (1 y 2), qué familias lo explican y si son campañas puntuales o actividad sostenida (3), a qué familias establecidas prestarles más atención este mes (4), y qué formatos bloquear o inspeccionar para cada una (5: `elf` apunta a dispositivos Linux/IoT; `exe` o `zip`, a usuarios de Windows).

**Para leer bien los datos:** la fecha es cuándo URLhaus *registró* la URL, no cuándo ocurrió el ataque; el último mes está incompleto (por eso 1, 3 y 4 usan solo meses completos); la familia es aproximada, porque sale de los tags que carga cada reportero.

Para la entrega: sacar una captura del dashboard y guardarla en `docs/tablero.png` del repo.